# 🎬 Netflix Movie Recommendation System
## Content-Based Filtering with TF-IDF & Cosine Similarity
---
### 📌 Introduction

Recommendation systems are one of the most impactful applications of machine learning in everyday life.
From Netflix to Spotify, these systems help users discover new content they are likely to enjoy.

This notebook implements a **Content-Based Filtering (CBF)** recommendation system using the **Netflix Movies Dataset**.
Instead of relying on what other users liked (collaborative filtering), CBF recommends items **similar in content** to what a user has already watched.

### 🎯 Business Questions
1. Given a movie a user has watched and rated, **which other Netflix movies best match their taste** based on genre, country, and description?
2. How can we **build a user preference profile** from their watch history using a rating-weighted feature vector?
3. How can we **serve personalized recommendations to multiple users** simultaneously using their watch and rating history?

### 🧠 Methodology
Both Single and Multi-User modes follow the **same rating-weighted pipeline**:

```
Item-Feature Matrix (TF-IDF + Genre + Country)
        × User Rating
        ↓
Item-Feature Matrix with Rating
        ↓
Sum per feature → User Feature Vector
        ↓
Normalize (÷ total rating sum)
        ↓
Cosine Similarity vs unseen movies
        ↓
Top-N Recommendations
```

| Step | Technique |
|------|----------|
| Text Feature Extraction | TF-IDF Vectorizer on `description` |
| Categorical Features | Multi-hot encoding on `genre` and `country` |
| Feature Weighting | Item-Feature Matrix × User Rating |
| User Profile | Normalized sum → User Feature Vector |
| Similarity | Cosine Similarity |
| Recommendation | Single User & Multi-User CBF |


---
## 1. 📦 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

print('Libraries loaded successfully!')

---
## 2. 📂 Load Dataset

In [ ]:
df = pd.read_csv('../netflix_titles.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
df.info()

---
## 3. 🔍 Exploratory Data Analysis (EDA)

### 3.1 Missing Values

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()
print(df.isnull().sum())

### 3.2 Content Type Distribution (Movies vs TV Shows)

In [ ]:
type_counts = df['type'].value_counts()
plt.figure(figsize=(6, 4))
sns.barplot(x=type_counts.index, y=type_counts.values, palette='Set2')
plt.title('Content Type Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
print(type_counts)

### 3.3 Top 10 Countries by Movie Production

In [ ]:
movies_df = df[df['type'] == 'Movie'].copy()
country_counts = movies_df['country'].dropna().str.split(', ').explode().value_counts().head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=country_counts.values, y=country_counts.index, palette='Blues_r')
plt.title('Top 10 Countries by Movie Production on Netflix')
plt.xlabel('Number of Movies')
plt.tight_layout()
plt.show()

### 3.4 Top 10 Genres

In [ ]:
genre_col = 'listed_in' if 'listed_in' in movies_df.columns else 'genre'
genre_counts = movies_df[genre_col].dropna().str.split(', ').explode().value_counts().head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='Reds_r')
plt.title('Top 10 Movie Genres on Netflix')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

### 3.5 Description Word Count Distribution

In [ ]:
movies_df['desc_word_count'] = movies_df['description'].dropna().apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 4))
sns.histplot(movies_df['desc_word_count'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Description Word Count (Movies)')
plt.xlabel('Word Count')
plt.tight_layout()
plt.show()

---
## 4. 🔧 Feature Engineering

### 4.1 Filter Movies Only & Handle Missing Values

In [ ]:
movies = df[df['type'] == 'Movie'].copy().reset_index(drop=True)
genre_col = 'listed_in' if 'listed_in' in movies.columns else 'genre'

movies['description'] = movies['description'].fillna('')
movies['country'] = movies['country'].fillna('Unknown')
movies[genre_col] = movies[genre_col].fillna('Unknown')

print(f'Total movies: {len(movies)}')
movies[['title', genre_col, 'country', 'description']].head()

### 4.2 TF-IDF on Description

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_matrix = tfidf.fit_transform(movies['description'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Sample features: {tfidf.get_feature_names_out()[:20]}')

### 4.3 Multi-Hot Encoding for Genre & Country

In [ ]:
mlb_genre = MultiLabelBinarizer()
genre_matrix = mlb_genre.fit_transform(movies[genre_col].str.split(', '))

mlb_country = MultiLabelBinarizer()
country_matrix = mlb_country.fit_transform(movies['country'].str.split(', '))

print(f'Genre matrix shape:   {genre_matrix.shape}')
print(f'Country matrix shape: {country_matrix.shape}')

### 4.4 Combine All Features

In [ ]:
feature_matrix = sp.hstack([
    tfidf_matrix,
    sp.csr_matrix(genre_matrix),
    sp.csr_matrix(country_matrix)
]).toarray()  # Convert to dense for easy rating multiplication

print(f'Combined feature matrix shape: {feature_matrix.shape}')

# Build title → index mapping
movies = movies.reset_index(drop=True)
title_to_idx = pd.Series(movies.index, index=movies['title'].str.lower()).drop_duplicates()
print(f'Total indexed movies: {len(title_to_idx)}')

---
## 5. 📐 Similarity Measure

We use **Cosine Similarity** to measure how similar two feature vectors are:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

- Result = **0** → no similarity
- Result = **1** → identical content

---
## 6. 🎯 Content-Based Filtering

Both Single User and Multi-User modes use the **same rating-weighted pipeline** from the lecture:

| Step | Operation |
|------|----------|
| 1 | Item-Feature Matrix (binary: TF-IDF + genre + country) |
| 2 | Multiply each movie's feature row × user rating |
| 3 | Sum all weighted rows → raw User Feature Vector |
| 4 | Normalize ÷ total rating sum → final User Feature Vector |
| 5 | Cosine Similarity: User Feature Vector vs all unwatched movies |
| 6 | Sort descending → Top-N Recommendations |


In [ ]:
def build_user_feature_vector(watched_ratings):
    '''
    Build a rating-weighted, normalized user feature vector.

    Parameters:
        watched_ratings : dict {title (str): rating (int/float)}

    Pipeline (matches lecture slide):
        1. Item-Feature Matrix × rating  → weighted rows
        2. Sum all weighted rows          → raw User Feature Vector
        3. Normalize ÷ total rating sum  → normalized User Feature Vector
    '''
    weighted_sum = np.zeros(feature_matrix.shape[1])
    total_rating = 0

    for title, rating in watched_ratings.items():
        if rating > 0 and title.lower() in title_to_idx:
            idx = title_to_idx[title.lower()]
            # Step 1 & 2: feature row × rating, then accumulate
            weighted_sum += feature_matrix[idx] * rating
            total_rating += rating

    if total_rating == 0:
        return weighted_sum

    # Step 3: normalize
    return weighted_sum / total_rating


def recommend(watched_ratings, top_n=5):
    '''
    Recommend unseen movies based on rating-weighted user feature vector.

    Parameters:
        watched_ratings : dict {title (str): rating (int/float)}
        top_n           : number of recommendations to return
    '''
    user_vec = build_user_feature_vector(watched_ratings)
    sim_scores = cosine_similarity([user_vec], feature_matrix)[0]

    # Exclude already watched movies
    watched_indices = set(
        title_to_idx[t.lower()] for t in watched_ratings
        if t.lower() in title_to_idx and watched_ratings[t] > 0
    )

    scores_filtered = [
        (i, s) for i, s in enumerate(sim_scores)
        if i not in watched_indices
    ]
    scores_sorted = sorted(scores_filtered, key=lambda x: x[1], reverse=True)[:top_n]

    indices = [s[0] for s in scores_sorted]
    scores  = [round(s[1], 4) for s in scores_sorted]

    result = movies[['title', genre_col, 'country']].iloc[indices].copy()
    result['similarity_score'] = scores
    return result.reset_index(drop=True)

### 6.1 Single User Recommendation

The user provides **one or more movies they've watched along with their rating (1–10)**.
The system builds their preference profile using the rating-weighted feature vector pipeline.

In [ ]:
# Single User: provide title(s) + rating(s)
single_user_ratings = {
    'Inception': 9,
}

print('=== Single User Recommendations ===')
print(f'Based on: {single_user_ratings}\n')
recommend(single_user_ratings, top_n=5)

In [ ]:
# Try with multiple watched movies (still single user)
single_user_ratings_multi = {
    'Inception':    9,
    'Bird Box':     7,
    'The Irishman': 8,
}

print('=== Single User (Multi-Watch History) Recommendations ===')
print(f'Based on: {single_user_ratings_multi}\n')
recommend(single_user_ratings_multi, top_n=5)

### 6.2 Multiple User Recommendation

We simulate a **User-Item Rating Matrix** (same as the lecture notebook).
Each user's rating-weighted feature vector is built independently — no influence between users.

In [ ]:
# User-Item Rating Matrix (0 = not watched)
user_item_data = {
    'user':          ['User 1', 'User 2', 'User 3', 'User 4'],
    'Inception':     [9, 8, 0, 0],
    'The Irishman':  [0, 7, 9, 0],
    'Bird Box':      [8, 0, 7, 9],
    'Mank':          [0, 0, 8, 7],
}
df_user_items = pd.DataFrame(user_item_data).set_index('user')
df_user_items

In [ ]:
# Generate recommendations for all users
for user in df_user_items.index:
    watched_ratings = {
        title: rating
        for title, rating in df_user_items.loc[user].items()
        if rating > 0
    }
    print(f'\n🎬 Recommendations for {user}')
    print(f'   Watched & rated: {watched_ratings}')
    print(recommend(watched_ratings, top_n=3).to_string(index=False))

---
## 7. 📊 Summary

- **Content-Based Filtering** was implemented using TF-IDF on movie descriptions combined with multi-hot encoded genre and country features.
- **Both Single User and Multi-User modes** use the same rating-weighted user feature vector pipeline, consistent with the lecture framework.
- **Cosine Similarity** was used to measure alignment between the user's preference profile and unseen movies.
- Key insight: Higher-rated genres/descriptions dominate the user feature vector, driving more accurate and personalized recommendations.
